# Tarea N°1 — Inteligencia Artificial
## Parte 1: Aprendizaje de Estructura de Red Bayesiana

**Dataset:** Adult Income (UCI Census Income Dataset)  
**Librería principal:** pgmpy  

### Descripción del dataset
El dataset *Adult Income* contiene ~48.000 registros del censo de EE.UU. con características sociodemográficas de individuos. El objetivo original es predecir si el ingreso anual de una persona supera los 50.000 USD. Tiene más de 14 columnas, de las cuales seleccionaremos 7 variables útiles para construir la red bayesiana.

## 1. Carga y Preprocesamiento del Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


df_raw = pd.read_csv('adult.csv')

# Renombrar columnas al formato esperado
df_raw.columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

print(f"Dimensiones originales: {df_raw.shape}")
df_raw.head()

Dimensiones originales: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [7]:
# Reemplazar '?' por NaN y eliminar filas con valores faltantes
df_raw.replace('?', np.nan, inplace=True)
df_raw.dropna(inplace=True)
print(f"Dimensiones tras limpiar NaN: {df_raw.shape}")

# Selección de 7 variables útiles
variables = ['age', 'education_num', 'occupation', 'workclass', 'hours_per_week', 'sex', 'income']
df = df_raw[variables].copy()

print("\nDistribución de tipos:")
print(df.dtypes)
print(f"\nFilas totales: {len(df)}")

Dimensiones tras limpiar NaN: (30162, 15)

Distribución de tipos:
age               int64
education_num     int64
occupation          str
workclass           str
hours_per_week    int64
sex                 str
income              str
dtype: object

Filas totales: 30162


In [ ]:
# ---- Discretización de variables continuas ----
# Las redes bayesianas con pgmpy requieren variables discretas/categóricas.

# age -> 3 grupos: joven (<=30), adulto (31-50), mayor (>50)
df['age'] = pd.cut(df['age'], bins=[0, 30, 50, 100],
                   labels=['joven', 'adulto', 'mayor'])

# education_num -> bajo (<=8), medio (9-12), alto (>12)
df['education_num'] = pd.cut(df['education_num'], bins=[0, 8, 12, 20],
                              labels=['bajo', 'medio', 'alto'])

# hours_per_week -> part_time (<40), full_time (40), overtime (>40)
df['hours_per_week'] = pd.cut(df['hours_per_week'], bins=[0, 39, 40, 100],
                               labels=['part_time', 'full_time', 'overtime'])

# Simplificar workclass a 3 categorías
def simplify_workclass(w):
    if w in ['Private']:
        return 'Privado'
    elif w in ['Self-emp-not-inc', 'Self-emp-inc']:
        return 'Independiente'
    else:
        return 'Gobierno_Otro'

df['workclass'] = df['workclass'].apply(simplify_workclass)

# Simplificar occupation a 4 grupos
def simplify_occupation(o):
    manual = ['Craft-repair', 'Farming-fishing', 'Handlers-cleaners', 'Machine-op-inspct', 'Transport-moving']
    white  = ['Adm-clerical', 'Exec-managerial', 'Prof-specialty', 'Sales', 'Tech-support']
    service= ['Other-service', 'Priv-house-serv', 'Protective-serv']
    if o in manual:
        return 'Manual'
    elif o in white:
        return 'Oficina'
    elif o in service:
        return 'Servicio'
    else:
        return 'Otro'

df['occupation'] = df['occupation'].apply(simplify_occupation)

# Limpiar income
df['income'] = df['income'].str.strip().str.replace('.', '', regex=False)
df['income'] = df['income'].map({'<=50K': 'bajo', '>50K': 'alto'})

# Convertir todo a string para pgmpy
df = df.astype(str)

print("\n--- Dataset preprocesado ---")
print(df.head(10))
print(f"\nDimensiones finales: {df.shape}")

In [ ]:
# Verificar cardinalidades
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

## 2. Aprendizaje de Estructura

### 2.1 Método Hill-Climbing

**¿Cómo funciona Hill-Climbing?**

Hill-Climbing (HC) es un algoritmo de búsqueda local heurística. Parte de un grafo vacío (sin aristas) y en cada iteración evalúa tres tipos de operaciones sobre los arcos del grafo dirigido:

- **Agregar** un arco entre dos nodos
- **Eliminar** un arco existente
- **Revertir** la dirección de un arco

Cada operación es evaluada mediante una función de puntuación (por defecto BIC — Bayesian Information Criterion o BDeu). Se elige la operación que más mejora el puntaje. El proceso se repite hasta que ninguna operación mejore el puntaje actual (convergencia local).

**Ventajas:** Escalable a datasets grandes.  
**Desventajas:** Puede quedar atrapado en óptimos locales.

In [ ]:
from pgmpy.estimators import HillClimbSearch, BicScore, BDeuScore
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination

# Aprendizaje de estructura con Hill-Climbing
scoring_method = BicScore(df)
hc = HillClimbSearch(df)

print("Ejecutando Hill-Climbing...")
best_model_hc = hc.estimate(scoring_method=scoring_method, max_iter=1000)

print("\n--- Estructura aprendida por Hill-Climbing ---")
print("Arcos encontrados:")
for edge in best_model_hc.edges():
    print(f"  {edge[0]} -> {edge[1]}")

In [ ]:
import networkx as nx

def plot_bn(edges, title):
    G = nx.DiGraph()
    G.add_edges_from(edges)
    pos = nx.spring_layout(G, seed=42, k=2)
    plt.figure(figsize=(8, 5))
    nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='steelblue', alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_color='white', font_size=9, font_weight='bold')
    nx.draw_networkx_edges(G, pos, arrowsize=20, edge_color='gray',
                           connectionstyle='arc3,rad=0.1', width=2)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_bn(best_model_hc.edges(), "Red Bayesiana — Hill-Climbing")

### 2.2 Búsqueda Exhaustiva (ExhaustiveSearch)

**¿Cómo funciona la Búsqueda Exhaustiva?**

La búsqueda exhaustiva evalúa **todos los DAGs (grafos dirigidos acíclicos) posibles** sobre el conjunto de variables, calculando la puntuación de cada uno y retornando el de mayor puntaje. Garantiza encontrar el óptimo global.

**Limitación:** El número de DAGs posibles crece **super-exponencialmente** con el número de nodos. Con $n$ variables, el número de DAGs sigue la secuencia OEIS A003024:  
- 4 nodos → 543 DAGs  
- 5 nodos → 29.281 DAGs  
- 7 nodos → ~2.000.000 DAGs  

Por esta razón, aplicaremos búsqueda exhaustiva solo sobre un subconjunto de **4–5 variables** representativas, justificando la selección.

In [ ]:
from pgmpy.estimators import ExhaustiveSearch

# Justificación de la reducción de variables:
# Con 7 variables la búsqueda exhaustiva requiere evaluar millones de DAGs,
# lo que es computacionalmente inviable en un entorno estándar.
# Seleccionamos las 5 variables con mayor relevancia para predecir 'income':
# education_num, occupation, hours_per_week, sex, income.
# Esto nos permite realizar la búsqueda exhaustiva en tiempo razonable
# y comparar la estructura obtenida con la de Hill-Climbing.

vars_exhaustive = ['education_num', 'occupation', 'hours_per_week', 'sex', 'income']
df_small = df[vars_exhaustive].copy()

print("Ejecutando Búsqueda Exhaustiva (puede tardar unos minutos)...")
es = ExhaustiveSearch(df_small, scoring_method=BicScore(df_small))
best_model_ex = es.estimate()

print("\n--- Estructura aprendida por Búsqueda Exhaustiva ---")
print("Arcos encontrados:")
for edge in best_model_ex.edges():
    print(f"  {edge[0]} -> {edge[1]}")

In [ ]:
plot_bn(best_model_ex.edges(), "Red Bayesiana — Búsqueda Exhaustiva (5 variables)")

## 3. Estimación de Parámetros (CPDs)

Una vez definida la estructura, estimamos las **Tablas de Probabilidad Condicional (CPTs)** de cada nodo usando **Maximum Likelihood Estimation (MLE)**. MLE estima cada probabilidad como la frecuencia relativa observada en los datos para cada combinación de valores padre-hijo.

In [ ]:
# ---- Parámetros Red Hill-Climbing ----
model_hc = BayesianNetwork(best_model_hc.edges())
model_hc.fit(df, estimator=MaximumLikelihoodEstimator)

print("=== Parámetros Red Hill-Climbing ===")
for cpd in model_hc.cpds:
    print(cpd)
    print()

In [ ]:
# ---- Parámetros Red Búsqueda Exhaustiva ----
model_ex = BayesianNetwork(best_model_ex.edges())
model_ex.fit(df_small, estimator=MaximumLikelihoodEstimator)

print("=== Parámetros Red Búsqueda Exhaustiva ===")
for cpd in model_ex.cpds:
    print(cpd)
    print()

## 4. Inferencias a Posteriori (Diagnóstico)

Usamos **Variable Elimination** para calcular probabilidades a posteriori dada evidencia observada.

### 4.1 Inferencias en Red Hill-Climbing

In [ ]:
infer_hc = VariableElimination(model_hc)

# Inferencia 1: ¿Cuál es la probabilidad de ingreso alto dado que
# la persona tiene educación alta y trabaja en Oficina?
print("=== INFERENCIA HC 1 ===")
print("Consulta: P(income | education_num=alto, occupation=Oficina)")
q1_hc = infer_hc.query(
    variables=['income'],
    evidence={'education_num': 'alto', 'occupation': 'Oficina'}
)
print(q1_hc)
print("\nInterpretación: Dado que la persona tiene educación alta y trabaja en Oficina,",
      "la probabilidad de que su ingreso sea alto es:",
      round(q1_hc.values[list(q1_hc.state_names['income']).index('alto')], 4))

In [ ]:
# Inferencia 2: ¿Cuál es la distribución de educación dado que
# la persona tiene ingreso alto y hace overtime?
print("=== INFERENCIA HC 2 ===")
print("Consulta: P(education_num | income=alto, hours_per_week=overtime)")
q2_hc = infer_hc.query(
    variables=['education_num'],
    evidence={'income': 'alto', 'hours_per_week': 'overtime'}
)
print(q2_hc)
print("\nInterpretación: Entre quienes tienen ingreso alto y trabajan más de 40 horas,",
      "la distribución sobre el nivel de educación es la indicada arriba.")

### 4.2 Inferencias en Red Búsqueda Exhaustiva

In [ ]:
infer_ex = VariableElimination(model_ex)

# Inferencia 1: ¿Cuál es la probabilidad de ingreso dado que
# la persona es mujer y trabaja part_time?
print("=== INFERENCIA EX 1 ===")
print("Consulta: P(income | sex=Female, hours_per_week=part_time)")
q1_ex = infer_ex.query(
    variables=['income'],
    evidence={'sex': 'Female', 'hours_per_week': 'part_time'}
)
print(q1_ex)
print("\nInterpretación: Para una mujer que trabaja menos de 40 horas semanales,",
      "la probabilidad de tener ingreso bajo es:",
      round(q1_ex.values[list(q1_ex.state_names['income']).index('bajo')], 4))

In [ ]:
# Inferencia 2: ¿Cuál es la distribución de ocupación dado que
# la persona tiene educación media e ingreso alto?
print("=== INFERENCIA EX 2 ===")
print("Consulta: P(occupation | education_num=medio, income=alto)")
q2_ex = infer_ex.query(
    variables=['occupation'],
    evidence={'education_num': 'medio', 'income': 'alto'}
)
print(q2_ex)
print("\nInterpretación: Entre las personas con educación media e ingreso alto,",
      "la distribución por tipo de ocupación es la indicada arriba.")

## 5. Generación de Datos Sintéticos

Usamos el modelo Hill-Climbing (que usa las 7 variables completas) para generar muestras sintéticas que aumenten el dataset en un 10%, 20% y 40%.

In [ ]:
n_original = len(df)
print(f"Tamaño original del dataset: {n_original}")

augmented_datasets = {}

for pct in [0.10, 0.20, 0.40]:
    n_synthetic = int(n_original * pct)
    print(f"\nGenerando {n_synthetic} filas sintéticas ({int(pct*100)}%)...")
    
    # Muestreo desde la red bayesiana ajustada
    df_synthetic = model_hc.simulate(n_samples=n_synthetic)
    
    # Concatenar con dataset original
    df_augmented = pd.concat([df, df_synthetic], ignore_index=True)
    augmented_datasets[pct] = df_augmented
    
    print(f"  Dataset aumentado al {int(pct*100)}%: {len(df_augmented)} filas")

## 6. Re-entrenamiento con Datasets Aumentados y Comparación

Para cada dataset aumentado, repetimos: aprendizaje de estructura (HC), estimación de parámetros e inferencias.

In [ ]:
results = {}

for pct, df_aug in augmented_datasets.items():
    label = f"+{int(pct*100)}%"
    print(f"\n{'='*55}")
    print(f"  Procesando dataset aumentado {label} ({len(df_aug)} filas)")
    print(f"{'='*55}")
    
    # Aprendizaje de estructura
    hc_aug = HillClimbSearch(df_aug)
    model_aug = hc_aug.estimate(
        scoring_method=BicScore(df_aug),
        max_iter=1000
    )
    
    print(f"\nArcos aprendidos ({label}):")
    for e in model_aug.edges():
        print(f"  {e[0]} -> {e[1]}")
    
    # Estimación de parámetros
    bn_aug = BayesianNetwork(model_aug.edges())
    bn_aug.fit(df_aug, estimator=MaximumLikelihoodEstimator)
    
    # Inferencia equivalente a la del dataset original
    infer_aug = VariableElimination(bn_aug)
    
    try:
        q_aug = infer_aug.query(
            variables=['income'],
            evidence={'education_num': 'alto', 'occupation': 'Oficina'}
        )
        prob = round(q_aug.values[list(q_aug.state_names['income']).index('alto')], 4)
        print(f"  P(income=alto | education=alto, ocupacion=Oficina): {prob}")
    except Exception as ex:
        prob = None
        print(f"  Inferencia no disponible (arco no presente): {ex}")
    
    results[label] = {
        'n_filas': len(df_aug),
        'n_arcos': len(model_aug.edges()),
        'arcos': list(model_aug.edges()),
        'P_income_alto': prob
    }

# Dataset original
infer_orig = VariableElimination(model_hc)
q_orig = infer_orig.query(
    variables=['income'],
    evidence={'education_num': 'alto', 'occupation': 'Oficina'}
)
prob_orig = round(q_orig.values[list(q_orig.state_names['income']).index('alto')], 4)
results['Original'] = {
    'n_filas': n_original,
    'n_arcos': len(model_hc.edges()),
    'arcos': list(model_hc.edges()),
    'P_income_alto': prob_orig
}

In [ ]:
# ---- Tabla comparativa ----
summary = pd.DataFrame(results).T[['n_filas', 'n_arcos', 'P_income_alto']]
summary.index.name = 'Dataset'
print("\n=== TABLA COMPARATIVA ===")
print(summary.to_string())

In [ ]:
# ---- Gráfico comparativo ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = list(summary.index)
n_arcos = summary['n_arcos'].values.astype(int)
p_vals = summary['P_income_alto'].values.astype(float)

axes[0].bar(labels, n_arcos, color=['steelblue', 'coral', 'mediumseagreen', 'goldenrod'])
axes[0].set_title('Número de arcos aprendidos por dataset', fontweight='bold')
axes[0].set_ylabel('N° arcos')
axes[0].set_xlabel('Dataset')

axes[1].plot(labels, p_vals, marker='o', linewidth=2, color='steelblue')
axes[1].set_title('P(income=alto | educ=alto, ocup=Oficina)', fontweight='bold')
axes[1].set_ylabel('Probabilidad')
axes[1].set_xlabel('Dataset')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('comparacion_datasets.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Análisis Comparativo de Resultados

> **Nota:** El análisis de resultados que sigue debe ser completado por el estudiante con los valores reales obtenidos en la ejecución del notebook.

### Estructura de la red
- **Hill-Climbing vs Búsqueda Exhaustiva:** Comparar los arcos obtenidos por ambos métodos. ¿Coinciden las dependencias principales? ¿Existen diferencias en la dirección de algún arco?
- La búsqueda exhaustiva garantiza el óptimo global sobre el subconjunto de 5 variables, mientras que HC sobre las 7 puede quedar en un óptimo local.

### Efecto del aumento de datos sintéticos
- A medida que se agregan datos sintéticos generados desde la propia red, se espera que la **estructura aprendida se estabilice** o converja hacia la del dataset original, dado que los datos sintéticos reflejan las mismas distribuciones.
- El número de arcos puede variar levemente: más datos permiten detectar dependencias tenues que no eran estadísticamente significativas con el conjunto original.
- Las probabilidades a posteriori (inferencias) deberían mantenerse similares o reforzarse, ya que los datos adicionales provienen de la misma distribución.

### Limitaciones
- Los datos sintéticos no aportan nueva información real; su valor es principalmente para experimentos de robustez o técnicas de aumento de datos cuando los datos originales son escasos.